# Colab Setup - Clone from Git and Prepare Data

Run this notebook first in Colab to set up the environment.

# Clone your repository

In [1]:
# Clone your repository
REPO_URL = "https://github.com/Echo-Lee/RAG-embedding"  # CHANGE THIS

!git clone {REPO_URL}

# Extract repo name from URL
import os
repo_name = REPO_URL.split("/")[-1].replace(".git", "")
os.chdir(repo_name)

print(f"Current directory: {os.getcwd()}")
!ls -la

Cloning into 'RAG-embedding'...
remote: Enumerating objects: 46, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 46 (delta 7), reused 46 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (46/46), 34.13 KiB | 17.06 MiB/s, done.
Resolving deltas: 100% (7/7), done.
Current directory: /content/RAG-embedding
total 96
drwxr-xr-x 6 root root 4096 Mar  4 20:42 .
drwxr-xr-x 1 root root 4096 Mar  4 20:42 ..
-rw-r--r-- 1 root root 5985 Mar  4 20:42 COLAB_GIT_SETUP.md
-rw-r--r-- 1 root root 4183 Mar  4 20:42 COLAB_SETUP.md
drwxr-xr-x 2 root root 4096 Mar  4 20:42 experiments
drwxr-xr-x 8 root root 4096 Mar  4 20:42 .git
-rw-r--r-- 1 root root  720 Mar  4 20:42 .gitignore
-rw-r--r-- 1 root root 4406 Mar  4 20:42 GIT_PUSH_GUIDE.md
-rw-r--r-- 1 root root 3115 Mar  4 20:42 MIGRATION_NOTES.md
-rw-r--r-- 1 root root 4303 Mar  4 20:42 NEXT_STEPS.md
drwxr-xr-x 2 root root 4096 Mar  4 20:42 notebooks
-rw-r--r-- 1 root root 2988 Mar 

## 2. Mount Google Drive and Link Data

In [3]:
from google.colab import drive
drive.mount('/content/drive')

# Create data directories
!mkdir -p data/processed/hospital
!mkdir -p data/processed/corruption

# Option A: Link to Google Drive
# Upload your data to: Google Drive/Capstone-Data/
DRIVE_DATA_PATH = "/content/drive/MyDrive/Capstone-Data"  # CHANGE THIS

# Create symlinks
!ln -sf {DRIVE_DATA_PATH}/hospital/threads_with_summary.json data/processed/hospital/
!ln -sf {DRIVE_DATA_PATH}/corruption/emails_group_by_thread.json data/processed/corruption/

# Verify data files
print("\nData files:")
!ls -lhL data/processed/hospital/
!ls -lhL data/processed/corruption/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Data files:
total 28M
-rw------- 1 root root 28M Mar  4 19:53 threads_with_summary.json
total 3.3M
-rw------- 1 root root 3.3M Mar  4 20:09 emails_group_by_thread.json


## 3. Install Dependencies

In [ ]:
!pip install -q sentence-transformers faiss-cpu gradio pyyaml openai tqdm

# Check GPU
import torch
print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 4. Setup API Keys

In [ ]:
# Option A: Use Colab Secrets (Recommended)
# Go to: 🔑 Secrets (left sidebar) → Add new secret
# Add: AZURE_API_KEY and AZURE_ENDPOINT

from google.colab import userdata
import yaml

try:
    api_key = userdata.get('AZURE_API_KEY')
    endpoint = userdata.get('AZURE_ENDPOINT')
    print("✅ Retrieved API keys from Colab Secrets")
except:
    print("⚠️  Colab Secrets not found. Using manual input...")
    api_key = input("Enter Azure API Key: ")
    endpoint = input("Enter Azure Endpoint: ")

# Create config files from templates
for dataset in ['hospital', 'corruption']:
    template_path = f'experiments/{dataset}_base_template.yaml'
    config_path = f'experiments/{dataset}_base.yaml'

    with open(template_path, 'r') as f:
        config = yaml.safe_load(f)

    config['azure_api_key'] = api_key
    config['azure_endpoint'] = endpoint

    with open(config_path, 'w') as f:
        yaml.dump(config, f, default_flow_style=False)

    print(f"✅ Created {config_path}")

print("\n✅ Setup complete! You can now run launcher.ipynb")

## 5. Verify Setup

In [ ]:
# Check everything is ready
import os
from pathlib import Path

checks = [
    ("Hospital data", Path("data/processed/hospital/threads_with_summary.json").exists()),
    ("Corruption data", Path("data/processed/corruption/emails_group_by_thread.json").exists()),
    ("Hospital config", Path("experiments/hospital_base.yaml").exists()),
    ("Corruption config", Path("experiments/corruption_base.yaml").exists()),
    ("Source code", Path("src/config/config.py").exists()),
]

print("Setup Verification:")
print("=" * 50)
all_good = True
for name, status in checks:
    icon = "✅" if status else "❌"
    print(f"{icon} {name}")
    if not status:
        all_good = False

print("=" * 50)
if all_good:
    print("\n🎉 All checks passed! Ready to run launcher.ipynb")
else:
    print("\n⚠️  Some checks failed. Please fix before proceeding.")

## Next Steps

1. ✅ This setup is complete
2. Open `notebooks/launcher.ipynb`
3. Set MODE and DATASET
4. Run all cells to build index and launch demo